# Dementia (1): standard microstate parameters

**Aim:**
Compare control (CN) vs. Alzheimer disease (AD) subjects using three classical microstate parameters computed by `mstsa.dur_occ_cov`:
- **duration**: mean sojourn time per microstate (ms),
- **occurrence**: onsets per second per microstate,
- **coverage**: fraction of total recording time spent in each microstate.

**Data**: Dementia EEG microstate sequences (Miltiadous et al.)

**Note:** We have excluded the third group (fronto-temporal dementia) from this analysis.

In [ ]:
# JupyterLite/Pyodide only: install the pure-Python mstsa build (no numba/C
# extensions -- see https://github.com/Frederic-vW/mstsa/tree/pyodide). Falls
# through silently on a normal Jupyter install, where mstsa is already present.
try:
    import piplite
    await piplite.install(
        "https://raw.githubusercontent.com/Frederic-vW/mstsa/pyodide/wheels/mstsa-0.4.3-py3-none-any.whl"
    )
except ImportError:
    pass


In [ ]:
import os

data_path = "data/CN_AD"  # only participants.tsv is bundled in this copy;
                           # per-subject caches are looked up by filename only
f_participants = "data/CN_AD/participants.tsv"

with open(f_participants, 'r') as fp:
    lines = fp.readlines()
lines = [l.strip().split('\t') for l in lines[1:]]
group_of = {l[0]: l[3] for l in lines}

# participants.tsv lists the full original dataset (all 3 groups); only a
# curated subset actually has data here. That subset is exactly what's been
# cached (data/cache/dur_occ_cov/), which this copy bundles instead of the
# raw MPILMBB/CN_AD .npy files -- so derive the real subject list from the
# cache directory itself, then split by group via participants.tsv.
cached_subjects = sorted(f.split('_')[0] for f in os.listdir("data/cache/dur_occ_cov")
                          if f.endswith('.npz'))
subjects_CN = [s for s in cached_subjects if group_of.get(s) == 'C']
subjects_AD = [s for s in cached_subjects if group_of.get(s) == 'A']
subjects = subjects_CN + subjects_AD

ms_files = {
    'CN': sorted([f"{data_path}/{subj}_EC_ms_K4.npy" for subj in subjects_CN]),
    'AD': sorted([f"{data_path}/{subj}_EC_ms_K4.npy" for subj in subjects_AD]),
}
print(f"CN: {len(ms_files['CN'])}")
print(f"AD: {len(ms_files['AD'])}")


## Compute duration, occurrence, coverage per subject

`mstsa.dur_occ_cov(x, fs=500)` returns a dict of individual sojourn durations per
microstate, plus per-microstate occurrence and coverage arrays; we keep the
per-subject *mean* duration for each microstate. Per-subject results are cached to
disk (`data/cache/dur_occ_cov/*.npz`).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import mstsa

K = 4
fs = 500  # Hz
ms_labels = ['A', 'B', 'C', 'D']

cache_dir = "data/cache/dur_occ_cov"
os.makedirs(cache_dir, exist_ok=True)

_KEYS = ['dur', 'occ', 'cov']


def _compute_subject(f, K, fs):
    x = np.load(f).astype(np.int32)
    dur_dict, occ, cov = mstsa.dur_occ_cov(x, fs=fs)
    dur = np.array([np.mean(dur_dict[k]) for k in range(K)])
    return dict(dur=dur, occ=occ, cov=cov)


def analyze_subject(f, K=K, fs=fs, cache_dir=cache_dir, force=False):
    cache_file = os.path.join(cache_dir, os.path.basename(f).replace('.npy', '_doc.npz'))
    if os.path.exists(cache_file) and not force:
        d = np.load(cache_file)
        if all(k in d.files for k in _KEYS):
            return {k: d[k] for k in _KEYS}
    res = _compute_subject(f, K, fs)
    np.savez(cache_file, **res)
    return res


def analyze_group(files, max_subjects=None, **kwargs):
    files = files[:max_subjects] if max_subjects else files
    results = []
    for i, f in enumerate(files):
        cache_file = os.path.join(cache_dir, os.path.basename(f).replace('.npy', '_doc.npz'))
        cached = os.path.exists(cache_file)
        t0 = time.time()
        res = analyze_subject(f, **kwargs)
        tag = "cached" if cached else f"{time.time() - t0:.1f}s"
        print(f"  [{i + 1}/{len(files)}] {os.path.basename(f)} ({tag})")
        results.append(res)
    return results

In [ ]:
max_subjects = None  # full population

group_results = {}
for group in ['CN', 'AD']:
    print(f"=== {group} ===")
    group_results[group] = analyze_group(ms_files[group], max_subjects=max_subjects)

## Statistics: CN vs. AD per microstate

For each measure (duration, occurrence, coverage) and each microstate (A-D),
compare groups:
- Shapiro-Wilk normality test
  - both groups are normally distributed
    - Levene's test
      - equal variance: t-test
      - unequal variance: Welch's test
  - at least one group not normally distributed: Mann-Whitney U test

In [ ]:
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu


def _stack(results, key):
    return np.array([r[key] for r in results])  # (n_subjects, K)


_MEASURES = [
    ('dur', 'Duration (ms)'),
    ('occ', 'Occurrence (events/s)'),
    ('cov', 'Coverage'),
]


def compare_groups(cn, ad, alpha=0.05):
    """Pick t-test vs. Mann-Whitney U from a Shapiro-Wilk normality check on each
    group (both must pass to use a t-test); if normal, a Levene test on variance
    homogeneity picks Welch's vs. Student's t-test."""
    p_norm_cn = shapiro(cn).pvalue
    p_norm_ad = shapiro(ad).pvalue
    normal = (p_norm_cn > alpha) and (p_norm_ad > alpha)
    if normal:
        equal_var = levene(cn, ad).pvalue > alpha
        stat, p = ttest_ind(cn, ad, equal_var=equal_var)
        test = "Welch's t" if not equal_var else "Student's t"
    else:
        stat, p = mannwhitneyu(cn, ad)
        test = 'Mann-Whitney U'
    return dict(test=test, statistic=stat, pvalue=p,
                normal_cn=p_norm_cn, normal_ad=p_norm_ad, normal=normal)


def sig_stars(p):
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'


stats_results = {}
for key, label in _MEASURES:
    cn_all = _stack(group_results['CN'], key)
    ad_all = _stack(group_results['AD'], key)
    stats_results[key] = []
    print(f"\n{label}")
    print(f"{'MS':>3s} {'CN mean':>9s} {'AD mean':>9s}  {'Shapiro CN':>11s} {'Shapiro AD':>11s} "
          f"{'test':>14s} {'p-value':>10s}  sig")
    for k in range(K):
        res = compare_groups(cn_all[:, k], ad_all[:, k])
        stats_results[key].append(res)
        print(f"{ms_labels[k]:>3s} {cn_all[:, k].mean():9.2f} {ad_all[:, k].mean():9.2f}  "
              f"{res['normal_cn']:11.4g} {res['normal_ad']:11.4g} {res['test']:>14s} "
              f"{res['pvalue']:10.4g}  {sig_stars(res['pvalue'])}")

## Graphs

Grouped boxplots (CN blue, AD red) per microstate, one panel per measure, with a
significance bracket over each CN/AD pair.

In [ ]:
def add_sig_bracket(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.0, c='k')
    ax.text((x1 + x2) / 2, y + h, text, ha='center', va='bottom', fontsize=8)


def plot_dur_occ_cov(group_results, stats_results):
    fig, axes = plt.subplots(3, 1, figsize=(9, 12), sharex=True)
    rng = np.random.default_rng(0)
    box_width = 0.3
    offsets = {'CN': -0.18, 'AD': 0.18}
    colors = {'CN': 'tab:blue', 'AD': 'tab:red'}
    x_pos = np.arange(K)

    for ax, (key, label) in zip(axes, _MEASURES):
        all_vals_by_k = []
        for group in ['CN', 'AD']:
            data = _stack(group_results[group], key)  # (n_subjects, K)
            positions = x_pos + offsets[group]
            bp = ax.boxplot([data[:, k] for k in range(K)], positions=positions,
                             widths=box_width, patch_artist=True, showfliers=False)
            for patch in bp['boxes']:
                patch.set_facecolor(colors[group])
                patch.set_alpha(0.4)
            for k in range(K):
                jitter = (rng.random(data.shape[0]) - 0.5) * 0.08
                ax.scatter(np.full(data.shape[0], positions[k]) + jitter, data[:, k],
                           color='k', s=8, zorder=3)
        all_vals_by_k = [np.concatenate([_stack(group_results[g], key)[:, k] for g in ['CN', 'AD']])
                         for k in range(K)]
        ymax = max(v.max() for v in all_vals_by_k)
        ymin = min(v.min() for v in all_vals_by_k)
        span = ymax - ymin
        bracket_y = ymax + 0.14 * span
        bracket_h = 0.05 * span
        ax.set_ylim(ymin - 0.10 * span, bracket_y + 5 * bracket_h)
        for k in range(K):
            p = stats_results[key][k]['pvalue']
            add_sig_bracket(ax, x_pos[k] + offsets['CN'], x_pos[k] + offsets['AD'],
                             bracket_y, bracket_h, sig_stars(p))

        ax.set_xlim(-0.6, K - 0.4)
        ax.set_ylabel(label)
        ax.set_title(label, loc='left', fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)

    axes[-1].set_xticks(x_pos)
    axes[-1].set_xticklabels(ms_labels)
    axes[-1].set_xlabel('Microstate')

    import matplotlib.patches as mpatches
    legend_handles = [mpatches.Patch(facecolor=colors[g], alpha=0.4, label=g) for g in ['CN', 'AD']]
    axes[0].legend(handles=legend_handles, loc='upper right', frameon=False)

    fig.suptitle(f'CN vs. AD: microstate duration, occurrence, coverage '
                 f'(n={len(group_results["CN"])} CN, {len(group_results["AD"])} AD)',
                 fontsize=12, y=1.005)
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/07_dur_occ_cov.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_dur_occ_cov(group_results, stats_results)

## First-order syntax: between-group differences

- We analyze conditional transition matrices of *jump sequences*, but avoiding 
the randomization test null hypothesis due to the reasons explained in `01_first_order_syntax_limitations.ipynb` 

**Procedure:**  
- For each subject, compute the jump sequence conditional transition matrix 
`T_jump = mstsa.tpm_cond(jump, K)`.
- off-diagonal entries are compared CN vs. AD cell-by-cell
- the choice of test (parametric vs. non-parametric) is made once for the whole 
matrix: t-test if every cell passes a Shapiro-Wilk normality check in both groups; 
if one or more cells fail normality, every cell is compared with Mann-Whitney U

In [ ]:
tpm_cache_dir = "data/cache/tpm_jump"
os.makedirs(tpm_cache_dir, exist_ok=True)


def _compute_tpm_subject(f, K):
    x = np.load(f).astype(np.int32)
    jump, _ = mstsa.embedded_process(x)
    T = mstsa.tpm_cond(jump, K)
    return T


def analyze_tpm_subject(f, K=K, cache_dir=tpm_cache_dir, force=False):
    cache_file = os.path.join(cache_dir, os.path.basename(f).replace('.npy', '_tpm.npz'))
    if os.path.exists(cache_file) and not force:
        d = np.load(cache_file)
        if 'T' in d.files:
            return d['T']
    T = _compute_tpm_subject(f, K)
    np.savez(cache_file, T=T)
    return T


def analyze_tpm_group(files, max_subjects=None, **kwargs):
    files = files[:max_subjects] if max_subjects else files
    Ts = []
    for i, f in enumerate(files):
        cache_file = os.path.join(tpm_cache_dir, os.path.basename(f).replace('.npy', '_tpm.npz'))
        cached = os.path.exists(cache_file)
        t0 = time.time()
        T = analyze_tpm_subject(f, **kwargs)
        tag = "cached" if cached else f"{time.time() - t0:.1f}s"
        print(f"  [{i + 1}/{len(files)}] {os.path.basename(f)} ({tag})")
        Ts.append(T)
    return np.array(Ts)  # (n_subjects, K, K)


tpm_results = {}
for group in ['CN', 'AD']:
    print(f"=== {group} ===")
    tpm_results[group] = analyze_tpm_group(ms_files[group])

In [ ]:
off_diag = ~np.eye(K, dtype=bool)
i_idx, j_idx = np.where(off_diag)

# Decide the test once for the whole matrix: normal only if every off-diagonal cell
# passes Shapiro-Wilk in both groups.
normal_matrix = all(
    shapiro(tpm_results['CN'][:, i, j]).pvalue > 0.05 and
    shapiro(tpm_results['AD'][:, i, j]).pvalue > 0.05
    for i, j in zip(i_idx, j_idx)
)
matrix_test = "Welch's t" if normal_matrix else 'Mann-Whitney U'
print(f"Matrix-wide test selection: {'all cells normal' if normal_matrix else 'at least one cell non-normal'}"
      f" -> using {matrix_test} for every off-diagonal cell\n")

tpm_pvalues = np.full((K, K), np.nan)
for i, j in zip(i_idx, j_idx):
    cn_cell = tpm_results['CN'][:, i, j]
    ad_cell = tpm_results['AD'][:, i, j]
    if normal_matrix:
        _, p = ttest_ind(cn_cell, ad_cell, equal_var=False)
    else:
        _, p = mannwhitneyu(cn_cell, ad_cell)
    tpm_pvalues[i, j] = p

tpm_mean = {g: tpm_results[g].mean(axis=0) for g in ['CN', 'AD']}
tpm_diff = tpm_mean['CN'] - tpm_mean['AD']

print(f"{'from\\to':>8s}" + "".join(f"{ms_labels[j]:>16s}" for j in range(K)))
for i in range(K):
    row = f"{ms_labels[i]:>8s}"
    for j in range(K):
        if i == j:
            cell = '--'
        else:
            cell = f"{sig_stars(tpm_pvalues[i, j]):>4s} (p={tpm_pvalues[i, j]:.2g})"
        row += f"{cell:>16s}"
    print(row)

In [ ]:
def _annotate_matrix(ax, M, fmt='{:.2f}', mask_diag=True):
    for i in range(K):
        for j in range(K):
            if mask_diag and i == j:
                continue
            ax.text(j, i, fmt.format(M[i, j]), ha='center', va='center', fontsize=9)


def _highlight_significant(ax, pvalues, alpha=0.05, color='k', lw=2.5):
    for i in range(K):
        for j in range(K):
            if i == j or np.isnan(pvalues[i, j]) or pvalues[i, j] >= alpha:
                continue
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                        edgecolor=color, linewidth=lw))


def plot_tpm_comparison(tpm_mean, tpm_diff, tpm_pvalues):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    off_diag_max = max(tpm_mean['CN'][off_diag].max(), tpm_mean['AD'][off_diag].max())
    for ax, group in zip(axes[:2], ['CN', 'AD']):
        M = np.ma.array(tpm_mean[group], mask=np.eye(K, dtype=bool))
        im = ax.imshow(M, vmin=0, vmax=off_diag_max, cmap='viridis', aspect='equal')
        _annotate_matrix(ax, tpm_mean[group], fmt='{:.2f}', mask_diag=True)
        _highlight_significant(ax, tpm_pvalues, color='red')
        ax.set_xticks(range(K)); ax.set_xticklabels(ms_labels)
        ax.set_yticks(range(K)); ax.set_yticklabels(ms_labels)
        ax.set_xlabel('To'); ax.set_ylabel('From')
        ax.set_title(f'{group}: mean jump T (n={tpm_results[group].shape[0]})', fontweight='bold')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = axes[2]
    diff_masked = np.ma.array(tpm_diff, mask=np.eye(K, dtype=bool))
    vmax = np.abs(diff_masked).max()
    im = ax.imshow(diff_masked, vmin=-vmax, vmax=vmax, cmap='RdBu_r', aspect='equal')
    _annotate_matrix(ax, tpm_diff, fmt='{:+.2f}', mask_diag=True)
    _highlight_significant(ax, tpm_pvalues, color='k')
    ax.set_xticks(range(K)); ax.set_xticklabels(ms_labels)
    ax.set_yticks(range(K)); ax.set_yticklabels(ms_labels)
    ax.set_xlabel('To'); ax.set_ylabel('From')
    ax.set_title('CN - AD difference\n(box = significant, ' + matrix_test + ')', fontweight='bold')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/07_tpm_jump_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_tpm_comparison(tpm_mean, tpm_diff, tpm_pvalues)

**Conclusions:**
- The two groups, CN and AD, can be differentiated quite easily, several microstate parameters show statistically significant differences
- How else can we characterize them? Instead of the 'atomic' view of individual maps and between-class transitions, we can try to characterize the time series _as a whole_, largely ignoring the individual labels. Indirectly, we hope that this characterization will not depend on our (uncertain) choice of $K$, at least qualitatively